In [1]:
!pip install transformers datasets accelerate peft torch

# Dataset

In [2]:
from datasets import load_dataset

# Load medical QA dataset
dataset = load_dataset("lavita/medical-qa-datasets", "med-qa-en-4options-source")

train_data = dataset["train"]
eval_data  = dataset["validation"]

print(f"Train samples: {len(train_data)}")
print(f"Eval samples:  {len(eval_data)}")

README.md: 0.00B [00:00, ?B/s]

med-qa-en-4options-source/train-00000-of(…):   0%|          | 0.00/7.70M [00:00<?, ?B/s]

med-qa-en-4options-source/test-00000-of-(…):   0%|          | 0.00/1.01M [00:00<?, ?B/s]

med-qa-en-4options-source/validation-000(…):   0%|          | 0.00/976k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1272 [00:00<?, ? examples/s]

Train samples: 10178
Eval samples:  1272


In [3]:
def convert_to_instruction(example):
    return {
        "instruction": example["question"],
        "input": "",
        "output": example["answer"]
    }

train_dataset = train_data.map(convert_to_instruction)
eval_dataset  = eval_data.map(convert_to_instruction)

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

Map:   0%|          | 0/1272 [00:00<?, ? examples/s]

# Train

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [5]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [6]:
from peft import get_peft_config, get_peft_model, LoraConfig
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

In [7]:
# Define the tokenization function
def tokenize_function(examples):
    # Combine instruction, input, and output for a complete text to tokenize
    full_texts = []
    for i in range(len(examples["instruction"])):
        text = f"Instruction: {examples['instruction'][i]}"
        if examples['input'][i]:
            text += f"\nInput: {examples['input'][i]}"
        text += f"\nOutput: {examples['output'][i]}"
        full_texts.append(text)

    tokenized_inputs = tokenizer(
        full_texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    # For causal language modeling, labels are usually the input_ids themselves
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].clone()
    return tokenized_inputs

# Apply tokenization to the datasets
tokenized_train_dataset = train_dataset.map(
    tokenize_function, batched=True, remove_columns=train_dataset.column_names
)
tokenized_eval_dataset = eval_dataset.map(
    tokenize_function, batched=True, remove_columns=eval_dataset.column_names
)

# Instantiate DataCollatorForLanguageModeling, required for training with causal LMs
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

Map:   0%|          | 0/1272 [00:00<?, ? examples/s]

In [8]:
training_args = TrainingArguments(
    output_dir="./med_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,  # Set to False as suggested by the error
    fp16=True # Enable mixed-precision training for faster speed
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator
)

In [10]:
trainer.train()

Step,Training Loss
10,2.116820
20,2.048409
30,2.013911
40,1.921897
50,1.890454
60,1.887529
70,1.946734
80,1.845065
90,1.865837
100,1.795242


TrainOutput(global_step=2545, training_loss=1.7758712817268896, metrics={'train_runtime': 2007.1402, 'train_samples_per_second': 5.071, 'train_steps_per_second': 1.268, 'total_flos': 1.1207239506591744e+16, 'train_loss': 1.7758712817268896, 'epoch': 1.0})

In [12]:
trainer.save_model("./med_model_final")

# Test

In [13]:
from transformers import pipeline

qa = pipeline(
    "text-generation",
    model="./med_model_final",
    tokenizer=tokenizer,
    max_length=128
)
print(qa("What are common early symptoms of pneumonia?"))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/96 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=2048) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'What are common early symptoms of pneumonia? How does pneumonia progress over time?\nHow do you know if you have a bacterial or viral infection?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress over time?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress over time?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress over time?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress over time?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress over time?\nWhat are the symptoms of a viral infection?\nWhat are common early symptoms of bacterial pneumonia?\nHow does pneumonia progress

In [14]:
prompt = "Instruction: What are common early symptoms of pneumonia?\nOutput:"
print(qa(prompt, max_new_tokens=100))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Instruction: What are common early symptoms of pneumonia?\nOutput: Sore throat, fever, cough, respiratory distress, wheezing, and difficulty talking.'}]


## Validate

In [15]:
import torch
from tqdm import tqdm

def evaluate_model(model, tokenizer, eval_dataset, max_samples=200):
    model.eval()
    correct = 0
    total = 0

    for example in tqdm(eval_dataset.select(range(max_samples))):
        question = example["instruction"]
        true_answer = example["output"].strip()

        prompt = f"Instruction: {question}\nOutput:"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False   # important for deterministic output
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only generated answer
        predicted_answer = generated_text.split("Output:")[-1].strip()

        if predicted_answer == true_answer:
            correct += 1

        total += 1

    accuracy = correct / total
    print(f"\nAccuracy: {accuracy * 100:.2f}%")

    return accuracy

evaluate_model(model, tokenizer, eval_dataset)

100%|██████████| 200/200 [03:25<00:00,  1.03s/it]


Accuracy: 0.50%


0.005

# Test model with Ui

In [16]:
!pip install transformers peft accelerate gradio torch

In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Base model (same one used during training)
base_model_name = "Qwen/Qwen2.5-0.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/med_model_final"
)

model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=896, out_fea

In [19]:
def generate_response(message, history):
    # Convert chat history into prompt format
    prompt = ""

    for user_msg, bot_msg in history:
        prompt += f"User: {user_msg}\nAssistant: {bot_msg}\n"

    prompt += f"User: {message}\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only latest assistant response
    response = response.split("Assistant:")[-1].strip()

    return response

In [20]:
import gradio as gradio

interface = gradio.ChatInterface(
    fn=generate_response,
    title="🩺 Medical AI Assistant",
    description="Fine-tuned medical QA model",
)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


In [21]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://f172c217f76e3ed7b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
